# Florida Building Code — Model Routing Benchmark Report

Reproducible analysis notebook. Reads raw per-request results from `results/*.jsonl` and the evaluation set from `data/fbc_eval_questions.csv` — nothing here is pre-aggregated or cached. Re-running this notebook end-to-end after re-running the harness (Lessons 1–6) regenerates every chart and number below from source.

Sections:
1. Load and merge results
2. Baseline hallucination rate per model class (cold, no grounding)
3. Grounded vs. ungrounded citation accuracy
4. Fine-tune breakeven curve
5. Latency / cost tradeoff by query category
6. Decision engine sanity checks

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = Path("../results")
DATA_DIR = Path("../data")

plt.rcParams["figure.dpi"] = 110

## 1. Load and merge results

Expects one `.jsonl` file per model-class run, each row matching the `InferenceResult` schema from Lessons 1–4, plus a `_grounded` suffix on `question_id` for grounded runs (Lesson 6). Also loads the gold evaluation set to compute correctness.

In [ ]:
def load_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"WARNING: {path} not found — run the corresponding lesson harness first.")
        return pd.DataFrame()
    rows = [json.loads(line) for line in path.open()]
    return pd.DataFrame(rows)

result_files = {
    "foundation": RESULTS_DIR / "foundation_raw.jsonl",
    "instruction_tuned": RESULTS_DIR / "instruction_tuned_raw.jsonl",
    "slm": RESULTS_DIR / "slm_raw.jsonl",
    "multimodal": RESULTS_DIR / "multimodal_raw.jsonl",
    "foundation_grounded": RESULTS_DIR / "foundation_grounded.jsonl",
    "instruction_tuned_grounded": RESULTS_DIR / "instruction_tuned_grounded.jsonl",
    "slm_grounded": RESULTS_DIR / "slm_grounded.jsonl",
}

frames = []
for label, path in result_files.items():
    df = load_jsonl(path)
    if not df.empty:
        df["run_label"] = label
        df["grounded"] = label.endswith("_grounded")
        frames.append(df)

results = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

gold_path = DATA_DIR / "fbc_eval_questions.csv"
gold = pd.read_csv(gold_path) if gold_path.exists() else pd.DataFrame()
if gold.empty:
    print(f"WARNING: {gold_path} not found — create the evaluation set from Lesson 1 first.")

results.head()

In [ ]:
# Join results to gold answers on the base question_id (strip _grounded suffix)
if not results.empty and not gold.empty:
    results["base_question_id"] = results["question_id"].str.replace("_grounded", "", regex=False)
    merged = results.merge(
        gold, left_on="base_question_id", right_on="question_id", suffixes=("", "_gold")
    )
    merged["citation_correct"] = merged["cited_section"] == merged["gold_section"]
else:
    merged = pd.DataFrame()

merged.head()

## 2. Baseline hallucination rate per model class (cold, no grounding)

The headline number from Lessons 1–3: what fraction of answers cite a wrong or nonexistent FBC/Naples section when the model has no retrieved context to work from?

In [ ]:
if not merged.empty:
    cold = merged[~merged["grounded"]]
    hallucination_rate = (
        cold.groupby("model_class")["citation_correct"]
        .apply(lambda s: 1 - s.mean())
        .rename("hallucination_rate")
        .sort_values(ascending=False)
    )
    display(hallucination_rate.to_frame())

    ax = hallucination_rate.plot(kind="bar", title="Cold Hallucination Rate by Model Class")
    ax.set_ylabel("Fraction of answers with wrong/invented section citation")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("No merged data yet — populate results/ and data/ from Lessons 1-3.")

### Breakdown by query category

Expect `jurisdiction_amendment` questions (Naples/Collier-specific) to show the highest hallucination rate across all model classes — these are the most under-represented in any model's pretraining corpus.

In [ ]:
if not merged.empty:
    cold = merged[~merged["grounded"]]
    pivot = (
        cold.groupby(["category", "model_class"])["citation_correct"]
        .mean()
        .unstack("model_class")
    )
    display(pivot)
    pivot.plot(kind="bar", title="Citation Accuracy by Category (cold)")
    plt.ylabel("Citation accuracy")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

## 3. Grounded vs. ungrounded citation accuracy

The core Lesson 6 result: how much does injecting the actual FBC/Naples passage as context improve citation accuracy per model class? Expect the SLM to show the largest relative gain.

In [ ]:
if not merged.empty:
    grounding_effect = (
        merged.groupby(["model_class", "grounded"])["citation_correct"]
        .mean()
        .unstack("grounded")
        .rename(columns={False: "ungrounded", True: "grounded"})
    )
    grounding_effect["delta"] = grounding_effect["grounded"] - grounding_effect["ungrounded"]
    display(grounding_effect.sort_values("delta", ascending=False))

    grounding_effect[["ungrounded", "grounded"]].plot(
        kind="bar", title="Citation Accuracy: Grounded vs. Ungrounded"
    )
    plt.ylabel("Citation accuracy")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 4. Fine-tune breakeven curve

Uses the calculator from Lesson 5, parameterized with cost-per-request figures measured in Section 5 below (not assumed) plus a fine-tuning cost estimate you supply.

In [ ]:
def breakeven_analysis(
    large_model_cost_per_req: float,
    finetune_one_time_cost: float,
    finetuned_model_cost_per_req: float,
    monthly_requests: int,
):
    months = list(range(1, 13))
    cumulative_large = [large_model_cost_per_req * monthly_requests * m for m in months]
    cumulative_finetuned = [
        finetune_one_time_cost + finetuned_model_cost_per_req * monthly_requests * m
        for m in months
    ]
    breakeven_month = next(
        (m for m, (l, f) in zip(months, zip(cumulative_large, cumulative_finetuned)) if f < l),
        None,
    )
    return months, cumulative_large, cumulative_finetuned, breakeven_month

# Pull measured foundation-model cost/request from Section 5 if available, else placeholder
if not merged.empty and "cost_usd" in merged.columns:
    measured_foundation_cost = merged.loc[merged["model_class"] == "foundation", "cost_usd"].mean()
else:
    measured_foundation_cost = 0.02  # placeholder — replace with your measured value

months, cum_large, cum_ft, breakeven_month = breakeven_analysis(
    large_model_cost_per_req=measured_foundation_cost,
    finetune_one_time_cost=1500.0,      # curation + QLoRA training run — adjust to your quote
    finetuned_model_cost_per_req=0.002, # amortized SLM inference cost — adjust to your measurement
    monthly_requests=50_000,
)

print(f"Breakeven month: {breakeven_month}")
plt.plot(months, cum_large, label="Foundation model (prompted)")
plt.plot(months, cum_ft, label="Fine-tuned SLM")
plt.xlabel("Month")
plt.ylabel("Cumulative cost ($)")
plt.title("Fine-Tune Breakeven — 50K requests/month")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Latency / cost tradeoff by query category

In [ ]:
if not merged.empty:
    summary = (
        merged.groupby("model_class")
        .agg(
            p50_latency_ms=("latency_ms", "median"),
            p95_latency_ms=("latency_ms", lambda s: s.quantile(0.95)),
            mean_cost_usd=("cost_usd", "mean"),
            citation_accuracy=("citation_correct", "mean"),
        )
        .reset_index()
    )
    display(summary)

    fig, ax = plt.subplots()
    for _, row in summary.iterrows():
        ax.scatter(row["p95_latency_ms"], row["mean_cost_usd"], s=200 * row["citation_accuracy"] + 20)
        ax.annotate(row["model_class"], (row["p95_latency_ms"], row["mean_cost_usd"]))
    ax.set_xlabel("p95 latency (ms)")
    ax.set_ylabel("Mean cost per request ($)")
    ax.set_title("Latency vs. Cost (bubble size = citation accuracy)")
    plt.tight_layout()
    plt.show()

    # Persist the summary table used for the write-up — derived, not hand-edited
    summary.to_csv(RESULTS_DIR / "summary_table.csv", index=False)
    print(f"Wrote {RESULTS_DIR / 'summary_table.csv'}")

## 6. Decision engine sanity checks

Runs the Lesson 7 `recommend_model_class` function against a few representative constraint scenarios and prints the result, so you can eyeball whether the recommendations match the evidence in Sections 2–5 before wiring up the Streamlit UI.

In [ ]:
import sys
sys.path.append("..")

try:
    from engine.router import recommend_model_class

    scenarios = [
        dict(query_category="numeric", latency_budget_ms=2000, cost_ceiling_usd=5.0, accuracy_floor=0.9),
        dict(query_category="jurisdiction_amendment", latency_budget_ms=2000, cost_ceiling_usd=5.0, accuracy_floor=0.9),
        dict(query_category="definitional", latency_budget_ms=500, cost_ceiling_usd=1.0, accuracy_floor=0.8),
    ]

    for scenario in scenarios:
        result = recommend_model_class(merged, **scenario)
        print(scenario, "->", result)
except ImportError:
    print("engine/router.py not implemented yet — build it in Lesson 7.")